# Obtendo dados de Obras Públicas

## Páginas de interesse

Obtemos os dados de obras públicas de Caruaru, acessando o Portal da Transparência e selecionando "Obras Públicas":

![obras-publicas](../assets/cityhall_assets/secao_obras_publicas_prefeitura_caruaru.png)

Seremos direcionados para seção de obras, um conjunto de *cards* contendo informações de cada obra fica disponível:

![obras-publicas-cards](../assets/cityhall_assets/pagina_obras_publicas_prefeitura_caruaru.png)

Ao clicar em qualquer card, seremos direcionados para o formulário com as informações detalhadas de cada obra:

![ficha_obra](../assets/cityhall_assets/conteudo_detalhado_obras_prefeitura.png)

Estes são os dados que nos interessa. Nosso objetivo é percorrer cada página de cards, acessando cada um e coletando as informações de seus formulários.

## Exploração dos elementos de interesse:

Nossa primeira abordagem é navegar pelos elementos de interesse utilizando o Inspector do navegador. Nessa etapa, é importante desativar o JavaScript da página para identificarmos os elementos que são carregados programaticamente. No nosso caso, os dados são carregados diretamente na estrutura da página, o que significa que podemos navegar pelas estruturas utilizando consultas XPath para coletar as informações desejadas.

![xpath](../assets/cityhall_assets/caminho_para_card_obras.png)

Por exemplo, podemos localizar todos links dos cards da página com a seguinte consulta (até o momento da escrita desse documento): 

```
//section[@class="groupBox contrast"]/a[contains(@class, "box status")]/@href
```

## Recuperando todas as urls

In [2]:
import requests 
from parsel import Selector

In [3]:
url_base = "https://caruaru.pe.gov.br/portal-da-transparencia/obras-publicas/"
response_base = requests.get(url_base)

In [4]:
response_base

<Response [200]>

In [5]:
html_base = response_base.text
selector_base = Selector(html_base)

In [6]:
urls_obras = selector_base.xpath('//section[@class="groupBox contrast"]/a[contains(@class, "box status")]/@href').getall()

urls_obras


['https://caruaru.pe.gov.br/obras/dispensa-003-2024-ct-014-2024-contratacao-direta-de-empresa-especializada-na-prestacao-dos-servicos-de-manejo-dos-residuos-solidos-urbanos-no-municipio-de-caruaru-pe/',
 'https://caruaru.pe.gov.br/obras/preg-eletr-041-2023-ct-040-2023-fornecimento-e-transporte-de-agua-bruta-com-servico-de-irrigacao-atraves-de-caminhoes-pipa-para-fins-de-irrigacao-de-pracas-parques-urbanos-jardins-canteiro/',
 'https://caruaru.pe.gov.br/obras/cp-012-2023-ct-042-2023-execucao-da-reforma-e-manutencao-dos-cemiterios-publicos-do-municipio-de-caruaru/',
 'https://caruaru.pe.gov.br/obras/cp-013-2023-ct-038-2023-execucao-de-servicos-de-conservacao-das-estradas-rurais-no-municipio-de-caruaru-pe/',
 'https://caruaru.pe.gov.br/obras/ct-150-2023-preg-eletr-008-2023-ata-reg-precos-013-2023-locacao-transporte-montagem-instalacao-e-manutencao-desmontagem-de-palco-camarote-camarim-pavilhao-arqui/',
 'https://caruaru.pe.gov.br/obras/tp-001-2023-mercado-de-carnes-2a-etapa-requalificacao

## Obtendo dados das Obras 

Abaixo segue o processo para extração dos dados:

In [7]:
# Seleciona uma obra como exemplo para extração dos dados
url_obra_exemplo = urls_obras[0]
response_obra = requests.get(url_obra_exemplo)
html_obra = response_obra.text
selector_obra = Selector(html_obra)

In [9]:
# Obtém as categorias principais do formulário
categorias_principais = selector_obra.xpath('//section[@class="description-obra"]//div[@class="group-title"]/text()').getall()

categorias_principais

['MODALIDADE | Nº DA LICITAÇÃO',
 'DESCRIÇÃO DA OBRA',
 'CONVÊNIO',
 'CONTRATADO',
 'CONTRATO',
 'ADITIVO',
 'DESPESAS DO EXERCÍCIO',
 'VALOR PAGO ACUMULADO',
 'SITUAÇÃO',
 'ETAPA DA OBRA',
 'PERCENTUAL CONCLUÍDO',
 'TODOS OS DOCUMENTOS']

In [11]:
# Obtém as subcategorias (chaves aninhadas) do formulário
subcategorias = selector_obra.xpath('//div[@class="row-cards"]/div[@class="row-card"]/div[@class="card-title"]/text()').getall()
subcategorias

['Nº',
 'Concedente',
 'CPF | CNPJ',
 'Razão Social',
 'Nº',
 'Data ínicio',
 'Prazo',
 'Valor Contratado (R$)',
 'Data Conclusão / Paralisação',
 'Prazo Aditado',
 'Valor Aditado Acumulado (R$)',
 'Valor Médio Acumulado (R$)',
 'Valor pago Acumulado no período (R$)',
 'Valor pago Acumulado no exercício (R$)']

In [12]:
# Obtém os valores das chaves principais e aninhadas
valores_extracao = selector_obra.xpath('//section[@class="description-obra"]/div[@class="row-details"]//div[@class="card-info"]/text()[normalize-space()]').getall()

valores_extracao

['\n                                    DISPENSA 003/2024                                 \n                                ',
 'DISPENSA 003/2024 - CT 014/2024 - CONTRATAÇÃO DIRETA DE EMPRESA ESPECIALIZADA NA PRESTAÇÃO DOS SERVIÇOS DE MANEJO DOS RESÍDUOS SÓLIDOS URBANOS NO MUNICÍPIO DE CARUARU-PE',
 '\n                                            -',
 '\n                                            -',
 '\n                                            35.474.949/0001-08',
 '\n                                            LOCAR SANEAMENTO AMBIENTAL LTDA',
 '\n                                            014/2024',
 '\n                                            03/04/2024',
 '\n                                            12 MESES ',
 '\n                                            R$ 48.396.075,12',
 '-',
 '\n                                            -',
 '\n                                            R$ 48.396.075,12',
 '\n                                            R$ 3.374.463,60 ',
 '\n 

In [14]:
import re

# Lista para armazenar os valores formatados
valores_formatados = []

# Formata os valores extraídos
for valor in valores_extracao:
    valor_formatado = re.sub(r"\s+", " ", valor).strip()
    valores_formatados.append(valor_formatado)

valores_formatados

['DISPENSA 003/2024',
 'DISPENSA 003/2024 - CT 014/2024 - CONTRATAÇÃO DIRETA DE EMPRESA ESPECIALIZADA NA PRESTAÇÃO DOS SERVIÇOS DE MANEJO DOS RESÍDUOS SÓLIDOS URBANOS NO MUNICÍPIO DE CARUARU-PE',
 '-',
 '-',
 '35.474.949/0001-08',
 'LOCAR SANEAMENTO AMBIENTAL LTDA',
 '014/2024',
 '03/04/2024',
 '12 MESES',
 'R$ 48.396.075,12',
 '-',
 '-',
 'R$ 48.396.075,12',
 'R$ 3.374.463,60',
 'R$ 772.033,10',
 'R$ 772.033,10',
 'R$ 772.033,10',
 'em Andamento',
 'SERVIÇO CONTÍNUO',
 '1.60%']

In [15]:
# Obtém documentos e seus links
documentos = selector_obra.xpath('//div[@class="card-info card-info--docs"]/div[@class="attachment"]//p[@class="cardTitle"]/text()').getall()
links_documentos = selector_obra.xpath('//div[@class="card-info card-info--docs"]/div[@class="attachment"]//div[@class="button"]/a/@href').getall()

# Obtém a chave e valores de localização geográfica da obra
chave_mapa_obra = selector_obra.xpath('//section[@class="map-obra"]//div[@class="map-obra_title"]/p/text()').get()
valores_mapa_obra = selector_obra.xpath('//section[@class="map-obra"]//iframe/@src').getall()


In [23]:
# Estrutura dos dados

estrutura_formulario = {
    categorias_principais[0]: valores_formatados[0],
    categorias_principais[1]: valores_formatados[1],
    categorias_principais[2]: {
        subcategorias[0]: valores_formatados[2],
        subcategorias[1]: valores_formatados[3],
    },
    categorias_principais[3]: {
        subcategorias[2]: valores_formatados[4],
        subcategorias[3]: valores_formatados[5],
    },
    categorias_principais[4]: {
        subcategorias[4]: valores_formatados[6],
        subcategorias[5]: valores_formatados[7],
        subcategorias[6]: valores_formatados[8],
        subcategorias[7]: valores_formatados[9],
        subcategorias[8]: valores_formatados[10],
    },
    categorias_principais[5]: {
        subcategorias[9]: valores_formatados[11],
        subcategorias[10]: valores_formatados[12],
    },
    categorias_principais[6]: {
        subcategorias[11]: valores_formatados[13],
        subcategorias[12]: valores_formatados[14],
        subcategorias[13]: valores_formatados[15],
    },
    categorias_principais[7]: valores_formatados[16],
    categorias_principais[8]: valores_formatados[17],
    categorias_principais[9]: valores_formatados[18],
    categorias_principais[10]: valores_formatados[19],
    categorias_principais[11]: {
        documentos[0].strip(): links_documentos[0].strip(),
        documentos[1].strip(): links_documentos[1].strip(),
    },
    chave_mapa_obra: valores_mapa_obra,
}

estrutura_formulario


{'MODALIDADE | Nº DA LICITAÇÃO': 'DISPENSA 003/2024',
 'DESCRIÇÃO DA OBRA': 'DISPENSA 003/2024 - CT 014/2024 - CONTRATAÇÃO DIRETA DE EMPRESA ESPECIALIZADA NA PRESTAÇÃO DOS SERVIÇOS DE MANEJO DOS RESÍDUOS SÓLIDOS URBANOS NO MUNICÍPIO DE CARUARU-PE',
 'CONVÊNIO': {'Nº': '-', 'Concedente': '-'},
 'CONTRATADO': {'CPF | CNPJ': '35.474.949/0001-08',
  'Razão Social': 'LOCAR SANEAMENTO AMBIENTAL LTDA'},
 'CONTRATO': {'Nº': '014/2024',
  'Data ínicio': '03/04/2024',
  'Prazo': '12 MESES',
  'Valor Contratado (R$)': 'R$ 48.396.075,12',
  'Data Conclusão / Paralisação': '-'},
 'ADITIVO': {'Prazo Aditado': '-',
  'Valor Aditado Acumulado (R$)': 'R$ 48.396.075,12'},
 'DESPESAS DO EXERCÍCIO': {'Valor Médio Acumulado (R$)': 'R$ 3.374.463,60',
  'Valor pago Acumulado no período (R$)': 'R$ 772.033,10',
  'Valor pago Acumulado no exercício (R$)': 'R$ 772.033,10'},
 'VALOR PAGO ACUMULADO': 'R$ 772.033,10',
 'SITUAÇÃO': 'em Andamento',
 'ETAPA DA OBRA': 'SERVIÇO CONTÍNUO',
 'PERCENTUAL CONCLUÍDO': '1.60%

## Script completo

In [22]:
import re
import requests
from parsel import Selector

# URL da página principal de obras públicas
url_base = "https://caruaru.pe.gov.br/portal-da-transparencia/obras-publicas/"
response_base = requests.get(url_base)
html_base = response_base.text
selector_base = Selector(html_base)

# Obtém as URLs das obras
urls_obras = selector_base.xpath('//section[@class="groupBox contrast"]/a[contains(@class, "box status")]/@href').getall()

# Seleciona uma obra como exemplo para extração dos dados
url_obra_exemplo = urls_obras[0]
response_obra = requests.get(url_obra_exemplo)
html_obra = response_obra.text
selector_obra = Selector(html_obra)

# Obtém as categorias principais do formulário
categorias_principais = selector_obra.xpath('//section[@class="description-obra"]//div[@class="group-title"]/text()').getall()

# Obtém as subcategorias (chaves aninhadas) do formulário
subcategorias = selector_obra.xpath('//div[@class="row-cards"]/div[@class="row-card"]/div[@class="card-title"]/text()').getall()

# Obtém os valores das chaves principais e aninhadas
valores_extracao = selector_obra.xpath('//section[@class="description-obra"]/div[@class="row-details"]//div[@class="card-info"]/text()[normalize-space()]').getall()

# Lista para armazenar os valores formatados
valores_formatados = []

# Formata os valores extraídos
for valor in valores_extracao:
    valor_formatado = re.sub(r"\s+", " ", valor).strip()
    valores_formatados.append(valor_formatado)

# Obtém documentos e seus links
documentos = selector_obra.xpath('//div[@class="card-info card-info--docs"]/div[@class="attachment"]//p[@class="cardTitle"]/text()').getall()
links_documentos = selector_obra.xpath('//div[@class="card-info card-info--docs"]/div[@class="attachment"]//div[@class="button"]/a/@href').getall()

# Obtém a chave e valores de localização geográfica da obra
chave_mapa_obra = selector_obra.xpath('//section[@class="map-obra"]//div[@class="map-obra_title"]/p/text()').get()
valores_mapa_obra = selector_obra.xpath('//section[@class="map-obra"]//iframe/@src').getall()


# Estrutura do formulário com os dados da obra
estrutura_formulario = {
    categorias_principais[0]: valores_formatados[0],
    categorias_principais[1]: valores_formatados[1],
    categorias_principais[2]: {
        subcategorias[0]: valores_formatados[2],
        subcategorias[1]: valores_formatados[3],
    },
    categorias_principais[3]: {
        subcategorias[2]: valores_formatados[4],
        subcategorias[3]: valores_formatados[5],
    },
    categorias_principais[4]: {
        subcategorias[4]: valores_formatados[6],
        subcategorias[5]: valores_formatados[7],
        subcategorias[6]: valores_formatados[8],
        subcategorias[7]: valores_formatados[9],
        subcategorias[8]: valores_formatados[10],
    },
    categorias_principais[5]: {
        subcategorias[9]: valores_formatados[11],
        subcategorias[10]: valores_formatados[12],
    },
    categorias_principais[6]: {
        subcategorias[11]: valores_formatados[13],
        subcategorias[12]: valores_formatados[14],
        subcategorias[13]: valores_formatados[15],
    },
    categorias_principais[7]: valores_formatados[16],
    categorias_principais[8]: valores_formatados[17],
    categorias_principais[9]: valores_formatados[18],
    categorias_principais[10]: valores_formatados[19],
    categorias_principais[11]: {
        documentos[0].strip(): links_documentos[0].strip(),
        documentos[1].strip(): links_documentos[1].strip(),
    },
    chave_mapa_obra: valores_mapa_obra,
}

estrutura_formulario

{'MODALIDADE | Nº DA LICITAÇÃO': 'DISPENSA 003/2024',
 'DESCRIÇÃO DA OBRA': 'DISPENSA 003/2024 - CT 014/2024 - CONTRATAÇÃO DIRETA DE EMPRESA ESPECIALIZADA NA PRESTAÇÃO DOS SERVIÇOS DE MANEJO DOS RESÍDUOS SÓLIDOS URBANOS NO MUNICÍPIO DE CARUARU-PE',
 'CONVÊNIO': {'Nº': '-', 'Concedente': '-'},
 'CONTRATADO': {'CPF | CNPJ': '35.474.949/0001-08',
  'Razão Social': 'LOCAR SANEAMENTO AMBIENTAL LTDA'},
 'CONTRATO': {'Nº': '014/2024',
  'Data ínicio': '03/04/2024',
  'Prazo': '12 MESES',
  'Valor Contratado (R$)': 'R$ 48.396.075,12',
  'Data Conclusão / Paralisação': '-'},
 'ADITIVO': {'Prazo Aditado': '-',
  'Valor Aditado Acumulado (R$)': 'R$ 48.396.075,12'},
 'DESPESAS DO EXERCÍCIO': {'Valor Médio Acumulado (R$)': 'R$ 3.374.463,60',
  'Valor pago Acumulado no período (R$)': 'R$ 772.033,10',
  'Valor pago Acumulado no exercício (R$)': 'R$ 772.033,10'},
 'VALOR PAGO ACUMULADO': 'R$ 772.033,10',
 'SITUAÇÃO': 'em Andamento',
 'ETAPA DA OBRA': 'SERVIÇO CONTÍNUO',
 'PERCENTUAL CONCLUÍDO': '1.60%